# Data Pre-processing

In [ ]:
import sklearn as sk
import pandas as pd

## Import Data

In [ ]:
df = pd.read_csv("US_Accidents_March23.csv")
len(df)

## Basic Data Preprocessing
To simplify your project a bit, use only the data instances that contain at most 1 missing value. According to our counts:
There are 3,554,549 data instances with no missing values:
There are 244,702 data instances with exactly one missing value.
Hence, your dataset for the project should contain 3,554,549 + 244,702 = 3,799,251

In [ ]:
df.dropna(thresh=len(df.columns) - 1, inplace=True)
len(df)

In [ ]:
df.drop(columns=['ID', 'Description', 'Country', 'Source', 'Turning_Loop'], inplace = True)
print(df.columns)

In [ ]:
basic_df = df.copy(deep = True)

# Handle datetime -> integer by extracting integer values
basic_df["Start_Time"] = pd.to_datetime(basic_df["Start_Time"], format="ISO8601")
basic_df["End_Time"] = pd.to_datetime(basic_df["End_Time"], format="ISO8601")

basic_df["Duration_Minutes"] = (
    basic_df["End_Time"] - basic_df["Start_Time"]
).dt.total_seconds() / 60.0
basic_df["Start_Hour"] = basic_df["Start_Time"].dt.hour
basic_df["Start_DayOfWeek"] = basic_df["Start_Time"].dt.dayofweek  # 0 = Monday, 6 = Sunday
basic_df["Start_Month"] = basic_df["Start_Time"].dt.month
basic_df.drop(columns=["Start_Time", "End_Time"], inplace=True)

In [ ]:
# Drop location cols other than lat/long
basic_df.drop(
    columns=[
        "City", "County", "Zipcode", "Street", "Airport_Code", "State"
    ],
    inplace=True,
)

In [ ]:
#Convert sunset measures
sunset_fields = ["Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight"]
for f in sunset_fields:
    basic_df[f] = basic_df[f].map({"Day": 1, "Night": 0}) # day = 1, night = 0

In [ ]:
# One-hot encode wind direction and weather condition as dense columns
weather_columns = ["Wind_Direction", "Weather_Condition"]
basic_df = pd.get_dummies(
    basic_df,
    columns=weather_columns,
    dummy_na=True,
)

In [ ]:
#Drop timezone, weather
basic_df.drop(columns = ["Timezone", "Weather_Timestamp"], inplace = True)

In [ ]:
# Fill missing numeric values one column at a time to avoid allocating another full-size frame.
for column in basic_df.columns:
    missing = basic_df[column].isna()
    if missing.any():
        fill_value = basic_df[column].median()
        if pd.isna(fill_value):
            fill_value = 0
        basic_df[column] = basic_df[column].fillna(fill_value)

In [ ]:
basic_df.info()

In [ ]:
basic_df.head()

## Advanced Preprocessing 

TODO

# Export

In [ ]:
%pip install pyarrow

basic_df.to_parquet(
    "basic_processed_data.parquet",
    engine="pyarrow",
    compression="zstd",
    index=True
)